## EXPLORATION GOLD DATASET

In [1]:
#import pandas as pd
#import sqlalchemy as sa
#from indusense.db.session import create_postgres_engine
#from indusense.db.models import GoldMachineHourlyFeature

#engine = create_postgres_engine()
#stmt = sa.select(GoldMachineHourlyFeature)

#gold_df = pd.read_sql(stmt, engine)
#gold_df.head()
    

###############

import pandas as pd 

gold_df = pd.read_csv("C:\\Formation\\gold_dataset\\gold_dataset_20260526-104215.csv")
print (gold_df.head())

print (gold_df.isna().sum())
print(gold_df[gold_df.isna().any(axis=1)].head())

  machine_code         window_start           window_end  temp_mean_6h  \
0      MACH-04  2025-01-10 00:00:00  2025-01-10 01:00:00         37.51   
1      MACH-12  2025-01-10 10:00:00  2025-01-10 11:00:00         46.00   
2      MACH-02  2025-01-10 19:00:00  2025-01-10 20:00:00         42.34   
3      MACH-03  2025-01-12 05:00:00  2025-01-12 06:00:00         39.89   
4      MACH-05  2025-02-10 04:00:00  2025-02-10 05:00:00         33.11   

   temp_max_6h  temp_std_6h  pressure_mean_6h  pressure_max_6h  \
0        37.51          NaN               NaN              NaN   
1        46.00          NaN               NaN              NaN   
2        42.34          NaN               NaN              NaN   
3        39.89          NaN               NaN              NaN   
4        33.11          NaN               NaN              NaN   

   pressure_std_6h  temp_mean_12h  ...  type_arret_urgence_count_prev_24h  \
0              NaN          37.51  ...                                  0   
1   

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

ModuleNotFoundError: No module named 'sklearn'

## Recommandation pour action sur les NaN
Si une colonne est critique et que les valeurs manquantes sont rares, dropna() est souvent acceptable.
Si les valeurs manquantes sont fréquentes, mieux vaut fillna() avec une moyenne/médiane ou une méthode d’interpolation.
Toujours commencer par inspecter.

En résumé : inspecter les NaN, puis choisir dropna, fillna, ou interpolate selon si vous pouvez supprimer les lignes ou si vous devez conserver et estimer les valeurs manquantes.

Supprimer les lignes contenant des NaN
gold_df_clean = gold_df.dropna()

Remplacer les NaN par une valeur fixe
gold_df_filled = gold_df.fillna(0)

Interpolation si les données sont temporelles / ordonnées
gold_df = gold_df.interpolate()

In [ ]:
# Résumé concis des NaN
nan_summary = gold_df.isna().sum()
cols_with_nan = nan_summary[nan_summary > 0]

print(f"Dimensions: {gold_df.shape}")
print(f"\nColonnes avec NaN: {len(cols_with_nan)}")
if len(cols_with_nan) > 0:
    print(cols_with_nan)
    print(f"\nPourcentage de NaN (max): {(cols_with_nan.max() / len(gold_df) * 100):.2f}%")
else:
    print("✓ Aucun NaN dans le dataset!")
    
nan_pct = (gold_df.isna().sum() / len(gold_df) * 100)
nan_pct[nan_pct > 0].sort_values(ascending=False)

Dimensions: (66678, 45)

Colonnes avec NaN: 18
temp_std_6h                          15
pressure_mean_6h                   1117
pressure_max_6h                    1117
pressure_std_6h                    1146
temp_std_12h                         15
pressure_mean_12h                  1079
pressure_max_12h                   1079
pressure_std_12h                   1106
temp_std_24h                         15
pressure_mean_24h                  1007
pressure_max_24h                   1007
pressure_std_24h                   1034
temp_trend_6h                       300
pressure_trend_6h                  1504
temp_zscore_24h                     120
incident_max_severity_prev_24h    63465
hours_since_last_incident          8851
ambient_humidity_pct              66678
dtype: int64

Pourcentage de NaN (max): 100.00%


ambient_humidity_pct              100.000000
incident_max_severity_prev_24h     95.181319
hours_since_last_incident          13.274243
pressure_trend_6h                   2.255617
pressure_std_6h                     1.718708
pressure_mean_6h                    1.675215
pressure_max_6h                     1.675215
pressure_std_12h                    1.658718
pressure_max_12h                    1.618225
pressure_mean_12h                   1.618225
pressure_std_24h                    1.550736
pressure_mean_24h                   1.510243
pressure_max_24h                    1.510243
temp_trend_6h                       0.449924
temp_zscore_24h                     0.179969
temp_std_6h                         0.022496
temp_std_24h                        0.022496
temp_std_12h                        0.022496
dtype: float64

Stratégie recommandée par type de colonne
1. Supprimer complètement ambient_humidity_pct (100% de NaN)
2. Pour incident_max_severity_prev_24h (95% de NaN) → Supprimer la colonne ou remplacer par 0 (pas d'incident)
3. Pour hours_since_last_incident (13% de NaN) → Remplacer par la médiane ou -1 (pas d'incident connu)
4. Pour les colonnes pressure_* et temp_* (1-2% de NaN) → Interpolation (données temporelles)


In [ ]:
# ===== MODULE DE NETTOYAGE DES NaN =====

print("=== NETTOYAGE DES NaN ===\n")

# 1. Supprimer les colonnes avec 100% de NaN
cols_to_drop = nan_pct[nan_pct == 100].index.tolist()
if cols_to_drop:
    print(f"1. Suppression de colonnes vides (100% NaN): {cols_to_drop}")
    gold_df = gold_df.drop(columns=cols_to_drop)
else:
    print("1. Aucune colonne avec 100% de NaN")

# 2. Traiter les colonnes avec > 50% de NaN
cols_high_nan = nan_pct[(nan_pct > 50) & (nan_pct < 100)].index.tolist()
if cols_high_nan:
    print(f"\n2. Colonnes avec >50% NaN: {cols_high_nan}")
    for col in cols_high_nan:
        print(f"   - {col}: remplissage avec -1")
        gold_df[col] = gold_df[col].fillna(-1)

# 3. Interpolation pour colonnes temporelles (pressure, temp)
cols_to_interpolate = [col for col in gold_df.columns 
                       if any(x in col for x in ['pressure', 'temp', 'trend'])]
cols_to_interpolate = [col for col in cols_to_interpolate if gold_df[col].isna().sum() > 0]
if cols_to_interpolate:
    print(f"\n3. Interpolation pour colonnes temporelles: {cols_to_interpolate}")
    for col in cols_to_interpolate:
        gold_df[col] = gold_df[col].interpolate(method='linear', limit_direction='both')

# 4. Remplissage avec médiane pour colonnes restantes
remaining_nan = gold_df.columns[gold_df.isna().any()].tolist()
if remaining_nan:
    print(f"\n4. Remplissage avec médiane: {remaining_nan}")
    for col in remaining_nan:
        median_val = gold_df[col].median()
        gold_df[col] = gold_df[col].fillna(median_val)
        print(f"   - {col}: {gold_df[col].isna().sum()} NaN restants")

# 5. Vérification finale
total_nan = gold_df.isna().sum().sum()
print(f"\n✓ RÉSULTAT FINAL: {total_nan} NaN restants")
print(f"Dimensions: {gold_df.shape}")


=== NETTOYAGE DES NaN ===

1. Suppression de colonnes vides (100% NaN): ['ambient_humidity_pct']

2. Colonnes avec >50% NaN: ['incident_max_severity_prev_24h']
   - incident_max_severity_prev_24h: remplissage avec -1

3. Interpolation pour colonnes temporelles: ['temp_std_6h', 'pressure_mean_6h', 'pressure_max_6h', 'pressure_std_6h', 'temp_std_12h', 'pressure_mean_12h', 'pressure_max_12h', 'pressure_std_12h', 'temp_std_24h', 'pressure_mean_24h', 'pressure_max_24h', 'pressure_std_24h', 'temp_trend_6h', 'pressure_trend_6h', 'temp_zscore_24h']

4. Remplissage avec médiane: ['hours_since_last_incident']
   - hours_since_last_incident: 0 NaN restants

✓ RÉSULTAT FINAL: 0 NaN restants
Dimensions: (66678, 44)


In [ ]:
# ===== ANALYSE CORRELATION ENTRE PRESSURE ET NaN =====

# Recharger le dataset original pour avoir les NaN
gold_df_original = pd.read_csv("C:\\Formation\\gold_dataset\\gold_dataset_20260526-104215.csv")

print("=== CORRÉLATION PRESSURE & NaN ===\n")

# 1. Identifier les colonnes pressure avec NaN
pressure_cols = [col for col in gold_df_original.columns if 'pressure' in col]
pressure_cols_with_nan = [col for col in pressure_cols if gold_df_original[col].isna().sum() > 0]

print(f"Colonnes pressure avec NaN: {pressure_cols_with_nan}\n")

# 2. Créer une colonne indicatrice: 1 si NaN dans pressure, 0 sinon
gold_df_original['has_pressure_nan'] = gold_df_original[pressure_cols_with_nan].isna().any(axis=1).astype(int)

# 3. Analyser les patterns des NaN
print(f"Nombre de lignes avec NaN dans pressure: {gold_df_original['has_pressure_nan'].sum()}")
print(f"Pourcentage: {gold_df_original['has_pressure_nan'].sum() / len(gold_df_original) * 100:.2f}%\n")

# 4. Vérifier si les NaN pressure coincident avec NaN d'autres colonnes
print("Corrélation entre NaN pressure et NaN d'autres colonnes:")
cols_nan_correlation = []
for col in gold_df_original.columns:
    if col not in pressure_cols and gold_df_original[col].isna().sum() > 0:
        # Comparer les lignes avec NaN
        both_nan = ((gold_df_original[col].isna() & gold_df_original['has_pressure_nan'].astype(bool)).sum())
        if both_nan > 0:
            pct = both_nan / gold_df_original['has_pressure_nan'].sum() * 100
            cols_nan_correlation.append((col, both_nan, pct))

if cols_nan_correlation:
    for col, count, pct in sorted(cols_nan_correlation, key=lambda x: x[2], reverse=True):
        print(f"  {col}: {count} lignes ({pct:.1f}% des lignes avec pressure NaN)")
else:
    print("  Aucune corrélation détectée - NaN pressure sont indépendants")

# 5. Vérifier si c'est un pattern temporel (lignes consécutives)
nan_indices = gold_df_original[gold_df_original['has_pressure_nan'] > 0].index.values
if len(nan_indices) > 0:
    print(f"\nPattern temporel des NaN pressure:")
    print(f"  Première occurrence: index {nan_indices[0]}")
    print(f"  Dernière occurrence: index {nan_indices[-1]}")
    gaps = nan_indices[1:] - nan_indices[:-1]
    print(f"  Écart moyen entre NaN: {gaps.mean():.0f} lignes")
    print(f"  Écart max: {gaps.max()} lignes")


=== CORRÉLATION PRESSURE & NaN ===

Colonnes pressure avec NaN: ['pressure_mean_6h', 'pressure_max_6h', 'pressure_std_6h', 'pressure_mean_12h', 'pressure_max_12h', 'pressure_std_12h', 'pressure_mean_24h', 'pressure_max_24h', 'pressure_std_24h', 'pressure_trend_6h']

Nombre de lignes avec NaN dans pressure: 1504
Pourcentage: 2.26%

Corrélation entre NaN pressure et NaN d'autres colonnes:
  ambient_humidity_pct: 1504 lignes (100.0% des lignes avec pressure NaN)
  incident_max_severity_prev_24h: 1396 lignes (92.8% des lignes avec pressure NaN)
  hours_since_last_incident: 347 lignes (23.1% des lignes avec pressure NaN)
  temp_trend_6h: 90 lignes (6.0% des lignes avec pressure NaN)
  temp_std_6h: 15 lignes (1.0% des lignes avec pressure NaN)
  temp_std_12h: 15 lignes (1.0% des lignes avec pressure NaN)
  temp_std_24h: 15 lignes (1.0% des lignes avec pressure NaN)
  temp_zscore_24h: 15 lignes (1.0% des lignes avec pressure NaN)

Pattern temporel des NaN pressure:
  Première occurrence: inde

# ===== SÉPARATION TRAIN / TEST =====

# Colonnes à exclure des features
COLS_META = ['machine_code', 'window_start', 'window_end', 'split_set']
COLS_LABELS = ['label_failure_next_6h', 'label_failure_next_12h',
               'label_failure_next_24h', 'label_failure_next_48h']

In [ ]:
# ===== SÉPARATION TRAIN / TEST =====

# Colonnes à exclure des features
COLS_META = ['machine_code', 'window_start', 'window_end', 'split_set']
COLS_LABELS = ['label_failure_next_6h', 'label_failure_next_12h',
               'label_failure_next_24h', 'label_failure_next_48h']

feature_cols = [c for c in gold_df.columns if c not in COLS_META + COLS_LABELS]

# Cible principale (modifiable selon l'horizon souhaité)
TARGET = 'label_failure_next_24h'

# Split basé sur la colonne existante
train_df = gold_df[gold_df['split_set'] == 'train']
test_df  = gold_df[gold_df['split_set'] == 'test']

X_train = train_df[feature_cols]
y_train = train_df[TARGET].astype(int)

X_test = test_df[feature_cols]
y_test = test_df[TARGET].astype(int)

# Résumé
print(f"Features        : {len(feature_cols)}")
print(f"Train           : {X_train.shape[0]} lignes  |  positifs: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Test            : {X_test.shape[0]} lignes  |  positifs: {y_test.sum()} ({y_test.mean()*100:.1f}%)")
# ===== ENTRAINEMENT DU MODÈLE =====

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Modèle — class_weight='balanced' car les pannes sont rares
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Prédictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Évaluation
print(classification_report(y_test, y_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC : {roc_auc_score(y_test, y_proba):.4f}")
print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred))
